# ChakraNet — Milestone 0: Repo & Data Plumbing
**SIH 2026 · PS 26078 · NCMRWF / MoES**

This notebook demonstrates:
1. Loading the 12 km multi-member ensemble forecast (23 members, 120h lead time) for **Cyclone Phailin (October 2013)**.
2. Generating an icosahedral geodesic mesh at **resolution level M9** (~16 km spacing).
3. Spherical regridding from the regular 12 km NWP grid onto the M9 mesh nodes.
4. Visualizing the raw ensemble mean precipitation and MSLP isobars.

In [ ]:
# Install dependencies in Google Colab if needed
import sys
if 'google.colab' in sys.modules:
    !pip install numpy scipy matplotlib xarray zarr shapely pyproj

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Ensure project root is in path
root_dir = Path.cwd().resolve()
if root_dir.name == 'notebooks':
    root_dir = root_dir.parent
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from config import EVENT_CONFIG, MESH_CONFIG
from data.loader import PhailinDataLoader
from data.mesh import IcosahedralMesh
from data.regrid import SphericalRegridder
from data.visualize import plot_milestone0_ensemble_mean

print(f"Event: {EVENT_CONFIG.name} ({EVENT_CONFIG.event_id})")
print(f"Ensemble Members: {EVENT_CONFIG.ensemble_members}")
print(f"Lead Times (h): {EVENT_CONFIG.lead_times_hours}")

### Step 1: Load 12 km NWP Ensemble & Best Track

In [ ]:
loader = PhailinDataLoader()
dataset = loader.load_or_generate_ensemble()

print("Variables loaded:", list(dataset.keys()))
print("Precipitation shape (members, times, lat, lon):", dataset['tp'].shape)
print("Peak 24h precipitation in ensemble:", np.max(dataset['tp']), "mm")

### Step 2: Build M9 Icosahedral Mesh & Edge Graph

In [ ]:
mesh = IcosahedralMesh(MESH_CONFIG)
print(f"M9 Mesh Nodes: {mesh.num_nodes}")
print(f"M9 Mesh Graph Edges: {mesh.num_edges}")
print(f"Average connectivity per node: {mesh.num_edges / mesh.num_nodes:.2f} neighbors")

### Step 3: Spherical Regridding to Mesh

In [ ]:
regridder = SphericalRegridder(mesh, dataset['lats'], dataset['lons'])
ens_mean_tp = np.mean(dataset['tp'][:, 9], axis=0)  # Landfall time index 9 (T+108h)
mesh_tp = regridder.grid_to_mesh(ens_mean_tp)

print(f"Regular Grid Shape: {ens_mean_tp.shape}")
print(f"Regridded Mesh Nodes Shape: {mesh_tp.shape}")
print(f"Max precipitation on mesh: {np.max(mesh_tp):.2f} mm")

### Step 4: Render Milestone 0 Verification Map

In [ ]:
saved_img = plot_milestone0_ensemble_mean()
from IPython.display import Image, display
display(Image(filename=str(saved_img)))